# Training Neural Networks

## Learning Objectives

By the end of this activity, you will be able to:

-   Qualitatively describe a version of the gradient descent algorithm for training neural networks.
-   Implement several pieces of training logic for neural networks, including gradient descent and early stopping.
-   Train a neural network for image classification on a small image data set.

***Note***: This activity involves working your way through a relatively complex body of code, most of which we’ve prepared for you. It’s very important to **not** modify any of the code we’ve provided.

## Runtime

This activity will be much more fast (and fun!) if you have access to a GPU. In Google Colab, you can enable a GPU by going to `Runtime -> Change runtime type` and selecting `GPU` as the hardware accelerator. If your code seems suspiciously slow, check to see if you’re using a GPU!

## Introduction

In this activity, we’ll go deeper into the study of training neural networks on a kind of prediction task that we haven’t seen before: *image classification*. The goal of image classification is to accept an image as input and output a label that describes the content of the image. For example, an algorithm that distinguishes between images of cats and images of dogs is performing image classification. Our example data set for this activity is a collection of images of hand signs representing letters of the American Sign Language (ASL) alphabet. The goal will be to train a neural network to recognize these hand signs. This kind of task can be useful for building systems that can translate image or video of ASL into text or speech.

The code below downloads a version of the data, which I initially accessed from [Kaggle](https://www.kaggle.com/datasets/datamunge/sign-language-mnist). The data consists of 28x28 pixel greyscale images of hand signs, along with labels indicating which letter of the alphabet each sign represents.

In [ ]:
# DO NOT MODIFY
import pandas as pd
from matplotlib import pyplot as plt
import torch
import torch.nn as nn
from torch.nn import Conv2d, MaxPool2d, Parameter, ReLU
import seaborn as sns
sns.set(style="whitegrid")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on {device}.")

train_url = "https://raw.githubusercontent.com/PhilChodrow/ml-notes/main/data/sign-language-mnist/sign_mnist_train.csv"
test_url = "https://raw.githubusercontent.com/PhilChodrow/ml-notes/main/data/sign-language-mnist/sign_mnist_test.csv"

df_train_and_val = pd.read_csv(train_url)
df_test          = pd.read_csv(test_url)

def prep_data(df):
    n, p = df.shape[0], df.shape[1] - 1
    y = torch.tensor(df["label"].values)
    X = df.drop(["label"], axis = 1)
    X = torch.tensor(X.values)
    X = torch.reshape(X, (n, 1, 28, 28))
    X = X / 255

    # important: move the data to GPU if available
    X, y = X.to(device), y.to(device)

    return X, y

df_train = df_train_and_val.sample(frac = 0.8, random_state = 42)
df_val   = df_train_and_val.drop(df_train.index)

X_train, y_train = prep_data(df_train)
X_val, y_val     = prep_data(df_val)
X_test, y_test   = prep_data(df_test)

We’ve split the data into three parts: we’ll use the training and validation sets for tuning our model, and we’ll use the test set at the very end to evaluate our final model.

Unlike most of our previous experience with machine learning algorithms, our data is no longer represented in tabular format (i.e. it’s not a data frame). Rather, the `X` variables are tensors (which you can think of as very similar to `numpy` arrays), with shapes of the format

`(image_index, num_channels, height, width)`

Here, `image_index` is the index of the image in the data set (you can think of this as a number, like “first image”, “second image”, etc.), `num_channels` is the number of color channels (1 for grayscale images, 3 for RGB images), and `height` and `width` are the dimensions of the image in pixels. So, if we inspect the shape of `X_train`:

In [ ]:
X_train.shape

we see that this data set contains 21,964 images, containing one color channel (since the images are greyscale), and the image shapes are 28 pixels by 28 pixels. On the other hand, `y_train` is just a tensor of shape `(21964,)`, containing the labels for each of the images in `X_train`. Each label in `y_train` is an integer which corresponds to a letter of the alphabet.

Here is an example of grabbing and visualizing a single image in the training data, with its true label shown in the title.

In [ ]:
# DO NOT MODIFY
ALPHABET = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

ix = 0  # index of image to visualize
im = X_train[ix,0].cpu() # 0th image in training set and 0th channel (only channel)
label = ALPHABET[y_train[ix]]

fig, ax = plt.subplots(figsize = (3,3))
ax.imshow(im, cmap = "gray")
ax.set(title = f"{label}")
ax.axis("off")

## Part A: Getting to Know the Data

### Exercise A1

Make a 4x4 grid of images from the training data set, with the corresponding labels shown in the titles as in the above part. It’s ok to choose just the first 16 images in the data set, or you can pick a random subset. You can use much of the code from above to help you.

In [ ]:
# TODO: Your code here

In modern deep learning, we almost never pass all of our training data to the model at once. Instead, we typically break up the data into small batches, and pass each batch to the model one at a time. There are both computational and statistical benefits to this approach. The code below creates data loaders for the training and validation data sets, with a batch size of 16.

In [ ]:
# DO NOT MODIFY
data_loader_train = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_train, y_train),
    batch_size = 16,
    shuffle = True
)

data_loader_val = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_val, y_val),
    batch_size = 16,
    shuffle = True
)

data_loader_test = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_test, y_test),
    batch_size = 16,
    shuffle = False
)

It is possible to grab a single batch of data from a data loader like this:

In [ ]:
X_batch, y_batch = next(iter(data_loader_train))

You can also loop through all batches in a data loader using a `for` loop, as shown below.

``` python
for X_batch, y_batch in data_loader_train:
    # ... do stuff with X_batch and y_batch ...
```

### Exercise A2

Please write a simple function called `num_batches` that takes a data loader as input and returns the number of batches in that data loader. Use it to compute the number of batches in both the training and validation data loaders.

In [ ]:
# TODO: Your code here

## Part B: Training a Neural Network

Now let’s start training some neural nets on this data set. Since we haven’t talked much about the architecture of neural networks, we’ll supply you with the two nets that we’re going to train today. The first is a simple linear model (which is a version of logistic regression), and the second is a small convolutional neural network (CNN).

**Please run the code cells below without modification**. We do encourage you to read the code and get a heuristic sense for how these models fit together!

Here’s our linear model:

In [ ]:
# DO NOT MODIFY
class LinearModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.pipeline = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 26)
        )

    # this is the customary name for the method that computes the scores
    # the loss is usually computed outside the model class during the training loop
    def forward(self, x):
        return self.pipeline(x)

And here is our convolutional neural network:

In [ ]:
# DO NOT MODIFY
class ConvNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.pipeline = torch.nn.Sequential(
            nn.Conv2d(1, 10, 5),
            ReLU(),
            nn.Conv2d(10, 5, 3),
            ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Flatten(),
            nn.Linear(605, 128),
            ReLU(),
            nn.Linear(128, 32),
            ReLU(),
            nn.Linear(32, len(ALPHABET))
        )

    def forward(self, x):
        return self.pipeline(x)

### Exercise B1

We’re going to want to visualize the training process of our models, including both the value of the loss and the accuracy on both the training and validation sets. Please implement a function called `visualize_metrics` that takes as input four lists: `train_losses`, `val_losses`, `train_acc`, and `val_acc`, and produces two plots side-by-side: one showing training and validation loss over epochs, and the other showing training and validation accuracy over epochs. Please make sure to include appropriate axis labels, titles, and legends in your plots. You may find it easier to use the `matplotlib` syntax `ax.plot(...)` syntax for this task, rather than Seaborn.

***Note***: Since accuracy always lies between 0 and 1, please make sure to set the y-axis limits of the accuracy plot so that the y-axis runs from 0 to 1. Please also set the y-axis limits of the loss plot so that the y-axis starts at 0.

In [ ]:
# TODO: Your code here

Test your implementation with the following code. Your plots don’t need to look fancy, but they should be well-labeled and easy to read.

In [ ]:
# DO NOT MODIFY
n = 100
training_loss = (1/n * torch.rand(n).cumsum(dim = 0)).tolist()[::-1]
validation_loss = (1/n * torch.rand(n).cumsum(dim = 0)).tolist()[::-1]
training_acc = (torch.rand(n).cumsum(dim = 0) / n).tolist()
validation_acc = (torch.rand(n).cumsum(dim = 0) / n).tolist()

visualize_metrics(training_loss, validation_loss, training_acc, validation_acc)

### Part C: Training Loops

Below, we’ve implemented functions to train and visualize the training process of these neural networks. These functions have options to substitute out some of PyTorch’s built-in functionality with your own implementations of gradient descent and early stopping, which we’ll fill in when we finally get to those parts of the activity.

In [ ]:
# DO NOT MODIFY
def epoch_loss(model, data_loader, loss_fn, optimizer = None, my_gradient_descent = False, training = False):
    """
    compute the value of the loss of the model on complete dataloader 
    optionally perform a training step using specified optimizer if training = True
    """

    # initialize the total running loss
    running_loss = 0.0

    # loop through the batches from the data loader
    for X_batch, y_batch in data_loader:
        
        # if training, reset the gradient information used in updates
        if training:
            optimizer.zero_grad()

        # compute the model outputs and compare them to the true labels
        # to calculate the loss
        outputs = model(X_batch)
        loss = loss_fn(outputs, y_batch)

        # update the running total loss: we'll average over all data points at the end
        running_loss += loss.item() * X_batch.size(0)

        # if training, perform a backward pass and update the model parameters
        if training:
            loss.backward()
            if my_gradient_descent:
                with torch.no_grad():  # disable gradient tracking
                    gradient_descent_step(model, alpha = 0.01)
            else:
                optimizer.step()
 
    # compute the average loss over the entire dataset
    epoch_loss = running_loss / len(data_loader.dataset)
    return epoch_loss

In [ ]:
# DO NOT MODIFY
def accuracy(model, data_loader):
    """
    compute the accuracy of the model on the complete dataloader
    """
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in data_loader:

            # get model outputs and convert to predicted class
            outputs = model(X_batch)
            _, predicted = torch.max(outputs.data, 1)

            # update total and correct counts
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

    return correct / total

In [ ]:
# DO NOT MODIFY
# this is our primary training loop
def train_model(model, data_loader_train, data_loader_val, my_gradient_descent = False, my_stopping_criterion = False, num_epochs = 25):
    """
    train a model on a training data loader, evaluating on a validation data loader, for a specified number of epochs, with options to use custom gradient descent and stopping criterion implementations
    """

    # set up the loss function and optimizer
    # after we implement gradient_descent, the only purpose of the optimizer is to perform some bookkeeping by calling optimizer.zero_grad() in epoch_loss
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr = 0.01)

    # initialize lists to store training and validation losses
    train_losses = []
    val_losses = []
    train_acc = []
    val_acc = []

    # loop over the specified number of epochs
    for epoch in range(num_epochs): 

        # training step, including loss computation on training data
        loss = epoch_loss(model, data_loader_train, loss_fn, optimizer, my_gradient_descent = my_gradient_descent, training = True)
        train_losses.append(loss)
        train_acc.append(accuracy(model, data_loader_train))

        # validation loss computation
        val_loss = epoch_loss(model, data_loader_val, loss_fn, training = False)
        val_losses.append(val_loss)
        val_acc.append(accuracy(model, data_loader_val))

        print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {loss:.4f}, Val Loss: {val_loss:.4f}")

        # optional stopping criterion
        if my_stopping_criterion:
            if stopping_criterion(train_losses, val_losses, train_acc, val_acc):
                print("Stopping criterion met, ending training.")
                break

    return train_losses, val_losses, train_acc, val_acc

Finally, the function below wraps the entire training process in a convenient way, including a visualization. This is where we finally use your `visualize_metrics` function from earlier.

In [ ]:
# DO NOT MODIFY
def train_and_visualize(model, my_gradient_descent = False, my_stopping_criterion = False, num_epochs = 10):
    """
    Convenience function for training a model and visualizing the training and validation losses and accuracies over epochs.
    """
    train_losses, val_losses, train_acc, val_acc = train_model(
        model,
        data_loader_train,
        data_loader_val,
        my_gradient_descent = my_gradient_descent,
        my_stopping_criterion = my_stopping_criterion,
        num_epochs = num_epochs
    )

    visualize_metrics(train_losses, val_losses, train_acc, val_acc)

### Exercise C1

To start, just try training the linear model using the built-in optimizer by running the code below. You don’t need to write any code for this part, just run the cell and observe the training process.

In [ ]:
# DO NOT MODIFY
model = LinearModel().to(device)
train_and_visualize(model, num_epochs = 10, my_gradient_descent = False)

### Exercise C2

Run the code block below to also train the convolutional neural network using the built-in optimizer. Again, you don’t need to write any code for this part, just run the cell and observe the training process. You may notice that this model takes longer to train than the linear model, since it has many more parameters.

In [ ]:
# DO NOT MODIFY
model = ConvNet().to(device)
train_and_visualize(model, num_epochs = 10, my_gradient_descent = False)

If you’re interested, you can run this code block to grab a sample image from the training set and see what label the trained model predicts for it. Running this cell multiple times will show you different images and predictions.

In [ ]:
# DO NOT MODIFY
ix = torch.randint(0, X_train.shape[0], (1,)).item()
sample_image = X_train[ix:ix+1]
output_scores = model(sample_image)
predictions = torch.argmax(output_scores, dim=1)
predicted_label = ALPHABET[predictions.item()]
true_label = ALPHABET[y_train[ix].item()]
plt.imshow(sample_image[0,0].cpu(), cmap='gray')
plt.title(f"True label: {true_label}\nPredicted Label: {predicted_label}")
plt.axis('off')
plt.show()

### Exercise C3

Please write a few sentences commenting on what you observe about the training processes of these two models. Which model seems to perform better? Why do you think that is? How do the loss and accuracy curves compare between the two models?

*[TODO: Your response here]*

## Part D: Custom Training Logic

Now it’s time to implement a few aspects of the neural network training process ourselves.

### Exercise D1: Gradient Descent

All modern methods for training neural networks are *gradient methods*. These methods work by making small adjustments to the model parameters in such a way that the loss function is decreased (usually). The simplest gradient method is called *gradient descent*. In gradient descent, we make the update

$$
\begin{aligned}
    w' \gets w - \alpha \times \delta w
\end{aligned}
$$

where $w$ is a model parameter (e.g. a weight or bias), $\delta w$ is the adjustment of the loss function with respect to that parameter, and $\alpha$ is a small positive number called the *learning rate*, and $w'$ is the updated value of the parameter which is used in the next iteration.

***Note***: The adjustment $\delta w$ is the entry of the *gradient* of the loss function corresponding to the parameter $w$. You can learn more about gradients in classes like Math 0223: Multivariable Calculus and CS 0451: Machine Learning.

In this exercise, we will implement a single step of gradient descent for a PyTorch model according to the formula above. Please implement a function called `gradient_descent_step` that takes as input a PyTorch model and a learning rate `alpha`, and updates each parameter of the model according to the gradient descent formula above.

***Implementation Notes***

-   In PyTorch, after calculating all the gradients using `loss.backward()`, the gradient for each parameter is stored in the `.grad` attribute of that parameter. For example, if `w` is a model parameter, then `w.grad` contains the adjustment of the loss with respect to `w`. `w.grad` is always the same shape as `w`: for example, if `w` is an array of shape `(10, 5)`, then `w.grad` is also an array of shape `(10, 5)`. You can therefore use vectorized operations to update the entire array `w` at once, without any loops.
-   You should use one `for` loop to loop through all parameters of the model, which you can do like this: `for w in model.parameters(): ...`
-   It is possible to implement `gradient_descent_step` in just three lines of code (including the function definition line) if you make use of vectorized operations.

In [ ]:
# TODO: Your code here

Test your implementation by retraining the models using your custom gradient descent function. You can do this by setting the `my_gradient_descent` argument to `True` in the `train_and_visualize` function calls below. Your results should look qualitative similar to those you obtained earlier, although you may find that your two models make different amounts of progress towards convergence depending on your stopping criterion.

In [ ]:
# DO NOT MODIFY
model = LinearModel().to(device)
train_and_visualize(model, num_epochs = 10, my_gradient_descent = True)

In [ ]:
# DO NOT MODIFY
model = ConvNet().to(device)
train_and_visualize(model, num_epochs = 10, my_gradient_descent = True)

### Exercise D2

You may have noticed that either or both of your models are training for “too long” – eventually more training steps just aren’t that helpful, or could even cause predictive performance to decrease due to overfitting. There are many ways to handle overfitting, but one way is to simply stop the model training process before the model starts to overfit. Please implement a `stopping_criterion` which causes your model to stop training once you think the training process “should be over.” Your implementation should accept the current vector of training losses, validation losses, training accuracies, and validation accuracies as input, and return `True` if the training process should stop, and `False` otherwise.

The actual logic is up to you! Stop training when the loss begins to increase? When the loss hasn’t improved for three consecutive epochs? When the accuracy reaches a certain threshold?

In [ ]:
# TODO: Your code here

Test your implementation by retraining the models using your custom stopping criterion as in the code blocks below. Your results should look qualitative similar to those you obtained earlier, although you may find that your two models make different amounts of progress towards convergence depending on your stopping criterion.

In [ ]:
# DO NOT MODIFY
linear_model = LinearModel().to(device)
train_and_visualize(linear_model, num_epochs = 25, my_gradient_descent = True, my_stopping_criterion = True)   

In [ ]:
# DO NOT MODIFY
CNN_model = ConvNet().to(device)
train_and_visualize(CNN_model, num_epochs = 25, my_gradient_descent = True, my_stopping_criterion = True)   

## Part E: Evaluation

Feel free to train your models again using either custom or built-in training logic until you’re satisfied with their performance. Then, evaluate your models on the test set by running the code below:

In [ ]:
# DO NOT MODIFY
print("Final evaluation on test set")
print("-----------------------")

print("Linear Model:")
acc_linear = accuracy(linear_model, data_loader_test)
print(f"Test set accuracy: {acc_linear:.4f}")
print()
print("Convolutional Neural Network:")
acc_cnn = accuracy(CNN_model, data_loader_test)
print(f"Test set accuracy: {acc_cnn:.4f}")

## Collaboration statement

In a markdown cell below, briefly list who or what you collaborated with and how. Cite any sources here or with relevant inline comments in your code. Acknowledge all contributors, both people and AI, and what portions of this notebook they contributed. You do not need to cite or acknowledge any material provided in the starter file(s).

## Submitting your notebook

You will simultaneously submit the following two files to the relevant assignment on [Gradescope](https://gradescope.com) via the “Upload option” (guide [here](https://guides.gradescope.com/hc/en-us/articles/21865616724749-Submitting-a-Code-assignment)). **Both files must be uploaded at the same time and the file names must match the specification exactly for the autotesting to run successfully.**

1.  `activity_training_neural_networks.ipynb`: Your completed IPython notebook. You can obtain this via the “File→Download→Download .ipynb” menu option in Colab.
2.  `activity_training_neural_networks.py`: Your completed IPython notebook as a Python file. You can obtain this via the “File→Download→Download .py” menu option in Colab. This file is used to provide line-level feedback on your submission.

You can submit multiple times, with only the most recent submission (before the final due date) assessed for credit. Gradescope will run a series of automated unit tests on your notebook (which may takes 10s of seconds depending on the complexity of the notebook). Note that the tests performed by Gradescope are limited. Passing all of the visible tests does not guarantee that your submission correctly satisfies all of the requirements of the assignment.